In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [38]:
dataset_path = r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw\x_data.xlsx'

In [59]:
main_df = pd.read_excel(r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw\x_data.xlsx')

In [63]:
def close_power_outage_duration(dataset_path):
    # Read dataset
    main_df = pd.read_excel(dataset_path)
    df = main_df.copy()
    
    # Ensure DATE column is datetime
    df['DATE'] = pd.to_datetime(df['DATE'])
    
    # Filter for specific day
    day_filter_data = df[df['DATE'].dt.date == pd.to_datetime("2025-06-25").date()]
    
    # Filter complaint types
    power_outage_data = day_filter_data[
        (day_filter_data['COMPLAINT TYPE'] == 'Power Outage') | 
        (day_filter_data['COMPLAINT TYPE'] == 'No Power Supply')
    ]
    
    # Convert time objects into datetime for subtraction
    def to_datetime(t):
        if pd.isnull(t):
            return None
        if isinstance(t, datetime):
            return t
        if isinstance(t, time):
            return datetime.combine(datetime.today(), t)
        return pd.to_datetime(t)
    
    # Apply conversion
    start = power_outage_data['COMPLAINT RECEIVED TIME'].apply(to_datetime)
    end   = power_outage_data['FINAL RESPONSE TIME'].apply(to_datetime)
    
    # Calculate difference in hours
    power_outage_data['DURATION_HOURS'] = (end - start).dt.total_seconds() / 3600
    
    # Round to nearest whole hour
    power_outage_data['DURATION_HOURS_ROUNDED'] = power_outage_data['DURATION_HOURS'].round()
    
    # Integer hours (floor)
    power_outage_data['DURATION_HOURS_INT'] = power_outage_data['DURATION_HOURS'].fillna(0).astype(int)
    
    # Select relevant columns
    close_open_hour = power_outage_data[['DIVISION','SUB-DIVISION', 'SHIFT DUTY', 'CLOSED/OPEN', 'DURATION_HOURS_INT']].copy()
    
    # Define classification function
    def classify_duration(x):
        if x <= 2:
            return "<2"
        elif 2 < x <= 4:
            return "2<4"
        elif 4 < x <= 8:
            return "4<8"
        elif x >= 8:
            return ">8"
        else:
            return None
    
    # Apply classification
    close_open_hour["DURATION_RANGE"] = close_open_hour["DURATION_HOURS_INT"].apply(classify_duration)

    pivot_df = pd.pivot_table(
        close_open_hour,
        values='DURATION_HOURS_INT',
        index=['DIVISION','SUB-DIVISION','SHIFT DUTY'],
        columns=['DURATION_RANGE','CLOSED/OPEN'],
        aggfunc='count',
        fill_value=0,
        margins=True,          
        margins_name='Grand Total'
    )
    
    return pivot_df

In [64]:
pivot_df = close_power_outage_duration(dataset_path)

In [65]:

pivot_df



DURATION_RANGE                               2<4    4<8     <2     >8  \
CLOSED/OPEN                               Closed Closed Closed Closed   
DIVISION     SUB-DIVISION      SHIFT DUTY                               
BARGARH      BARGARH-2         A               1      0      0      0   
BOLANGIR     BOLANGIR-2        B               0      0      1      0   
             BOLANGIR-3        B               0      0      1      0   
JHARSUGUDA   KUCHINDA          A               0      0      2      0   
                               B               0      0      1      0   
                               C               0      0      0      1   
KWED         JUNAGARH          A               0      0      2      0   
NUAPADA      KHARIAR           B               0      0      1      0   
                               C               0      0      1      0   
RAJGANGPUR   SDO-1, RAJGANPUR  B               0      0      1      0   
ROURKELA     SDO-6, BISRA      B               0      0      3      0   
SAMBALPUR-E  SDO-2 SAMBALPUR-E B               0      0      1      0   
TITLAGARH    PATNAGARH         B               0      0      2      0   
             TITLAGARH         A               0      0      1      1   
                               C               0      3      0      0   
Grand Total                                    1      3     17      2   

DURATION_RANGE                            Grand Total  
CLOSED/OPEN                                            
DIVISION     SUB-DIVISION      SHIFT DUTY              
BARGARH      BARGARH-2         A                    1  
BOLANGIR     BOLANGIR-2        B                    1  
             BOLANGIR-3        B                    1  
JHARSUGUDA   KUCHINDA          A                    2  
                               B                    1  
                               C                    1  
KWED         JUNAGARH          A                    2  
NUAPADA      KHARIAR           B                    1  
                               C                    1  
RAJGANGPUR   SDO-1, RAJGANPUR  B                    1  
ROURKELA     SDO-6, BISRA      B                    3  
SAMBALPUR-E  SDO-2 SAMBALPUR-E B                    1  
TITLAGARH    PATNAGARH         B                    2  
             TITLAGARH         A                    2  
                               C                    3  
Grand Total                                        23

In [46]:


# Flatten MultiIndex columns if needed
pivot_df.columns = [
    ' '.join([str(c) for c in col]).strip() 
    for col in pivot_df.columns.values
]
